In [11]:
import matplotlib.pyplot as plt
from sklearn import datasets
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
import numpy as np
import os

iris = datasets.load_iris()
x = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names
target = iris.target

In [12]:
# Conjunto de datos original
original = pd.DataFrame(data=x, columns=feature_names)
original['target'] = target

# Conjunto de datos estandarizado
x_scaled = StandardScaler().fit_transform(x)
estandarizados = pd.DataFrame(data=x_scaled, columns=feature_names)
estandarizados['target'] = target

# Conjunto de datos normalizado
x_minmax = MinMaxScaler().fit_transform(x)
normalizados = pd.DataFrame(data=x_minmax, columns=feature_names)
normalizados['target'] = target

# Creamos objetos PCA reutilizables
pca_95 = PCA(n_components=0.95)
pca_80 = PCA(n_components=0.80)

# PCA 95% sobre datos originales
x_pca95_original = pca_95.fit_transform(x)
columnas_pca95_original = [f'PC{i+1}' for i in range(x_pca95_original.shape[1])]
originalPCA95 = pd.DataFrame(data=x_pca95_original, columns=columnas_pca95_original)
originalPCA95['target'] = target

# PCA 80% sobre datos originales
x_pca80_original = pca_80.fit_transform(x)
columnas_pca80_original = [f'PC{i+1}' for i in range(x_pca80_original.shape[1])]
originalPCA80 = pd.DataFrame(data=x_pca80_original, columns=columnas_pca80_original)
originalPCA80['target'] = target

# PCA 95% sobre datos estandarizados
x_pca95_estandarizado = pca_95.fit_transform(x_scaled)
columnas_pca95_estandarizado = [f'PC{i+1}' for i in range(x_pca95_estandarizado.shape[1])]
estandarizadoPCA95 = pd.DataFrame(data=x_pca95_estandarizado, columns=columnas_pca95_estandarizado)
estandarizadoPCA95['target'] = target

# PCA 80% sobre datos estandarizados
x_pca80_estandarizado = pca_80.fit_transform(x_scaled)
columnas_pca80_estandarizado = [f'PC{i+1}' for i in range(x_pca80_estandarizado.shape[1])]
estandarizadoPCA80 = pd.DataFrame(data=x_pca80_estandarizado, columns=columnas_pca80_estandarizado)
estandarizadoPCA80['target'] = target

# PCA 95% sobre datos normalizados
x_pca95_normalizado = pca_95.fit_transform(x_minmax)
columnas_pca95_normalizado = [f'PC{i+1}' for i in range(x_pca95_normalizado.shape[1])]
normalizadoPCA95 = pd.DataFrame(data=x_pca95_normalizado, columns=columnas_pca95_normalizado)
normalizadoPCA95['target'] = target

# PCA 80% sobre datos normalizados
x_pca80_normalizado = pca_80.fit_transform(x_minmax)
columnas_pca80_normalizado = [f'PC{i+1}' for i in range(x_pca80_normalizado.shape[1])]
normalizadoPCA80 = pd.DataFrame(data=x_pca80_normalizado, columns=columnas_pca80_normalizado)
normalizadoPCA80['target'] = target

In [13]:

# Configuración de validación cruzada
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Diccionario con todos los conjuntos de datos
datasets = {
    'original': original,
    'estandarizados': estandarizados,
    'normalizados': normalizados,
    'originalPCA95': originalPCA95,
    'originalPCA80': originalPCA80,
    'estandarizadoPCA95': estandarizadoPCA95,
    'estandarizadoPCA80': estandarizadoPCA80,
    'normalizadoPCA95': normalizadoPCA95,
    'normalizadoPCA80': normalizadoPCA80
}

# Crear carpeta para guardar particiones
output_dir = 'particiones_cv'
os.makedirs(output_dir, exist_ok=True)

# Generar particiones para cada conjunto de datos
for dataset_name, dataset in datasets.items():
    print(f"Generando particiones para {dataset_name}...")
    
    # Crear carpeta específica para este dataset
    dataset_dir = os.path.join(output_dir, dataset_name)
    os.makedirs(dataset_dir, exist_ok=True)
    
    # Separar características y target
    X = dataset.drop('target', axis=1)
    y = dataset['target']
    
    # Generar las 5 particiones
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        # Crear carpeta para este fold
        fold_dir = os.path.join(dataset_dir, f'fold_{fold}')
        os.makedirs(fold_dir, exist_ok=True)
        
        # Separar train y test
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        
        # Guardar en CSV
        X_train.to_csv(os.path.join(fold_dir, 'X_train.csv'), index=False)
        X_test.to_csv(os.path.join(fold_dir, 'X_test.csv'), index=False)
        y_train.to_csv(os.path.join(fold_dir, 'y_train.csv'), index=False, header=True)
        y_test.to_csv(os.path.join(fold_dir, 'y_test.csv'), index=False, header=True)
        
        print(f"  Fold {fold}: Train={len(X_train)}, Test={len(X_test)}")

print("\n✓ Particiones generadas exitosamente en la carpeta 'particiones_cv'")
print(f"✓ Total: {len(datasets)} conjuntos de datos × {n_splits} folds = {len(datasets)*n_splits} conjuntos de particiones")

Generando particiones para original...
  Fold 1: Train=120, Test=30
  Fold 2: Train=120, Test=30
  Fold 3: Train=120, Test=30
  Fold 4: Train=120, Test=30
  Fold 5: Train=120, Test=30
Generando particiones para estandarizados...
  Fold 1: Train=120, Test=30
  Fold 2: Train=120, Test=30
  Fold 3: Train=120, Test=30
  Fold 4: Train=120, Test=30
  Fold 5: Train=120, Test=30
Generando particiones para normalizados...
  Fold 1: Train=120, Test=30
  Fold 2: Train=120, Test=30
  Fold 3: Train=120, Test=30
  Fold 4: Train=120, Test=30
  Fold 5: Train=120, Test=30
Generando particiones para originalPCA95...
  Fold 1: Train=120, Test=30
  Fold 2: Train=120, Test=30
  Fold 3: Train=120, Test=30
  Fold 4: Train=120, Test=30
  Fold 5: Train=120, Test=30
Generando particiones para originalPCA80...
  Fold 1: Train=120, Test=30
  Fold 2: Train=120, Test=30
  Fold 3: Train=120, Test=30
  Fold 4: Train=120, Test=30
  Fold 5: Train=120, Test=30
Generando particiones para estandarizadoPCA95...
  Fold 1: T

In [14]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import pickle
import json

# Definir los modelos a evaluar
models = {
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'NaiveBayes': GaussianNB(),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

# Función para calcular todas las métricas
def calcular_metricas(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted'),
        'recall': recall_score(y_true, y_pred, average='weighted'),
        'f1_score': f1_score(y_true, y_pred, average='weighted')
    }

# Estructura para guardar resultados
resultados = {dataset_name: {model_name: [] for model_name in models.keys()} 
              for dataset_name in datasets.keys()}

# Estructura para guardar probabilidades
probabilidades = {dataset_name: {model_name: [] for model_name in models.keys()} for dataset_name in datasets.keys()}

print("=" * 80)
print("Iniciando validación cruzada...")


Iniciando validación cruzada...


In [15]:
# Crear carpeta para guardar modelos y resultados
models_dir = 'modelos_cv'
os.makedirs(models_dir, exist_ok=True)

# Iterar sobre cada conjunto de datos
for dataset_name in datasets.keys():
    print(f"\n{'='*80}")
    print(f"DATASET: {dataset_name}")
    print(f"{'='*80}")
    
    dataset_dir = os.path.join(output_dir, dataset_name)
    
    # Iterar sobre cada fold
    for fold in range(1, n_splits + 1):
        print(f"\n--- Fold {fold}/{n_splits} ---")
        fold_dir = os.path.join(dataset_dir, f'fold_{fold}')
        
        # Cargar datos de entrenamiento y test
        X_train = pd.read_csv(os.path.join(fold_dir, 'X_train.csv'))
        X_test = pd.read_csv(os.path.join(fold_dir, 'X_test.csv'))
        y_train = pd.read_csv(os.path.join(fold_dir, 'y_train.csv')).values.ravel()
        y_test = pd.read_csv(os.path.join(fold_dir, 'y_test.csv')).values.ravel()
        
        # Entrenar y evaluar cada modelo
        for model_name, model in models.items():
            # Entrenar el modelo
            model.fit(X_train, y_train)
            
            # Realizar predicciones
            y_pred = model.predict(X_test)
            
            # Obtener probabilidades de pertenencia a cada clase
            y_proba = model.predict_proba(X_test)
            
            # Calcular métricas
            metricas = calcular_metricas(y_test, y_pred)
            resultados[dataset_name][model_name].append(metricas)
            
            # Almacenar probabilidades
            probabilidades[dataset_name][model_name].append({
                'fold': fold,
                'y_test': y_test.tolist(),
                'y_pred': y_pred.tolist(),
                'y_proba': y_proba.tolist()
            })
            
            # Guardar el modelo entrenado
            model_path = os.path.join(models_dir, dataset_name, model_name)
            os.makedirs(model_path, exist_ok=True)
            model_file = os.path.join(model_path, f'fold_{fold}.pkl')
            with open(model_file, 'wb') as f:
                pickle.dump(model, f)
            
            # Mostrar métricas y primeras probabilidades
            print(f"  {model_name}: F1={metricas['f1_score']:.4f}, Acc={metricas['accuracy']:.4f}")
            print(f"    Probabilidades (primeras 3 muestras):")
            for i in range(min(3, len(y_proba))):
                print(f"      Muestra {i+1}: {y_proba[i]} -> Predicción: {y_pred[i]}, Real: {y_test[i]}")

# Guardar probabilidades en archivo JSON
probabilidades_file = os.path.join(models_dir, 'probabilidades_cv.json')
with open(probabilidades_file, 'w') as f:
    json.dump(probabilidades, f, indent=4)

print(f"\n{'='*80}")
print("✓ Validación cruzada completada")
print(f"✓ Modelos guardados en '{models_dir}'")
print(f"✓ Probabilidades guardadas en '{probabilidades_file}'")
print(f"{'='*80}")



DATASET: original

--- Fold 1/5 ---
  KNN: F1=1.0000, Acc=1.0000
    Probabilidades (primeras 3 muestras):
      Muestra 1: [1. 0. 0.] -> Predicción: 0, Real: 0
      Muestra 2: [1. 0. 0.] -> Predicción: 0, Real: 0
      Muestra 3: [1. 0. 0.] -> Predicción: 0, Real: 0
  SVM: F1=1.0000, Acc=1.0000
    Probabilidades (primeras 3 muestras):
      Muestra 1: [0.96012224 0.02800189 0.01187587] -> Predicción: 0, Real: 0
      Muestra 2: [0.97544808 0.01446867 0.01008325] -> Predicción: 0, Real: 0
      Muestra 3: [0.97388702 0.01471115 0.01140184] -> Predicción: 0, Real: 0
  NaiveBayes: F1=0.9666, Acc=0.9667
    Probabilidades (primeras 3 muestras):
      Muestra 1: [1.00000000e+00 1.34269866e-15 2.90592795e-23] -> Predicción: 0, Real: 0
      Muestra 2: [1.00000000e+00 5.99153206e-17 4.60905627e-25] -> Predicción: 0, Real: 0
      Muestra 3: [1.00000000e+00 1.40418137e-16 1.86803790e-23] -> Predicción: 0, Real: 0
  RandomForest: F1=0.9666, Acc=0.9667  RandomForest: F1=0.9666, Acc=0.9667
  

In [16]:
# Calcular y mostrar estadísticas finales (media y desviación típica)
print("\n" + "="*80)
print("RESULTADOS FINALES - VALIDACIÓN CRUZADA (5 FOLDS)")
print("="*80)

# Crear estructura para guardar resultados finales
resultados_finales = {}

for dataset_name in datasets.keys():
    print(f"\n{dataset_name}:")
    print("-" * 80)
    resultados_finales[dataset_name] = {}
    
    for model_name in models.keys():
        # Extraer todas las métricas de los 5 folds
        metricas_folds = resultados[dataset_name][model_name]
        
        # Calcular media y desviación típica para cada métrica
        accuracy_values = [m['accuracy'] for m in metricas_folds]
        precision_values = [m['precision'] for m in metricas_folds]
        recall_values = [m['recall'] for m in metricas_folds]
        f1_values = [m['f1_score'] for m in metricas_folds]
        
        resultados_finales[dataset_name][model_name] = {
            'accuracy_mean': np.mean(accuracy_values),
            'accuracy_std': np.std(accuracy_values),
            'precision_mean': np.mean(precision_values),
            'precision_std': np.std(precision_values),
            'recall_mean': np.mean(recall_values),
            'recall_std': np.std(recall_values),
            'f1_mean': np.mean(f1_values),
            'f1_std': np.std(f1_values)
        }
        
        print(f"\n  {model_name}:")
        print(f"    Accuracy:  {np.mean(accuracy_values):.4f} ± {np.std(accuracy_values):.4f}")
        print(f"    Precision: {np.mean(precision_values):.4f} ± {np.std(precision_values):.4f}")
        print(f"    Recall:    {np.mean(recall_values):.4f} ± {np.std(recall_values):.4f}")
        print(f"    F1-Score:  {np.mean(f1_values):.4f} ± {np.std(f1_values):.4f}")

# Guardar resultados finales en JSON
resultados_file = os.path.join(models_dir, 'resultados_finales.json')
with open(resultados_file, 'w') as f:
    json.dump(resultados_finales, f, indent=4)

print(f"\n{'='*80}")
print(f"✓ Resultados finales guardados en '{resultados_file}'")
print("="*80)


RESULTADOS FINALES - VALIDACIÓN CRUZADA (5 FOLDS)

original:
--------------------------------------------------------------------------------

  KNN:
    Accuracy:  0.9667 ± 0.0298
    Precision: 0.9673 ± 0.0298
    Recall:    0.9667 ± 0.0298
    F1-Score:  0.9666 ± 0.0298

  SVM:
    Accuracy:  0.9667 ± 0.0298
    Precision: 0.9695 ± 0.0276
    Recall:    0.9667 ± 0.0298
    F1-Score:  0.9665 ± 0.0300

  NaiveBayes:
    Accuracy:  0.9467 ± 0.0400
    Precision: 0.9488 ± 0.0395
    Recall:    0.9467 ± 0.0400
    F1-Score:  0.9465 ± 0.0401

  RandomForest:
    Accuracy:  0.9467 ± 0.0267
    Precision: 0.9512 ± 0.0263
    Recall:    0.9467 ± 0.0267
    F1-Score:  0.9464 ± 0.0268

estandarizados:
--------------------------------------------------------------------------------

  KNN:
    Accuracy:  0.9667 ± 0.0365
    Precision: 0.9684 ± 0.0357
    Recall:    0.9667 ± 0.0365
    F1-Score:  0.9666 ± 0.0366

  SVM:
    Accuracy:  0.9533 ± 0.0452
    Precision: 0.9549 ± 0.0443
    Recall:  

In [17]:
# Crear tabla comparativa de F1-Score
print("\n" + "="*80)
print("TABLA COMPARATIVA - F1-SCORE MEDIO")
print("="*80)

# Crear DataFrame para visualización
comparison_data = []
for dataset_name in datasets.keys():
    for model_name in models.keys():
        f1_mean = resultados_finales[dataset_name][model_name]['f1_mean']
        f1_std = resultados_finales[dataset_name][model_name]['f1_std']
        comparison_data.append({
            'Dataset': dataset_name,
            'Modelo': model_name,
            'F1-Score': f1_mean,
            'Std': f1_std
        })

df_comparison = pd.DataFrame(comparison_data)

# Crear tabla pivote para mejor visualización
pivot_table = df_comparison.pivot(index='Dataset', columns='Modelo', values='F1-Score')
print("\n")
print(pivot_table.to_string())

# Encontrar mejor combinación
best_result = df_comparison.loc[df_comparison['F1-Score'].idxmax()]
print(f"\n{'='*80}")
print(f"MEJOR RESULTADO:")
print(f"  Dataset: {best_result['Dataset']}")
print(f"  Modelo: {best_result['Modelo']}")
print(f"  F1-Score: {best_result['F1-Score']:.4f} ± {best_result['Std']:.4f}")
print("="*80)


TABLA COMPARATIVA - F1-SCORE MEDIO


Modelo                   KNN  NaiveBayes  RandomForest       SVM
Dataset                                                         
estandarizadoPCA80  0.919732    0.892689      0.879833  0.906028
estandarizadoPCA95  0.919732    0.892689      0.879833  0.906028
estandarizados      0.966583    0.946533      0.946432  0.953216
normalizadoPCA80    0.953182    0.953182      0.926280  0.953250
normalizadoPCA95    0.926516    0.926106      0.946229  0.953250
normalizados        0.959933    0.946533      0.946432  0.953216
original            0.966650    0.946533      0.946432  0.966515
originalPCA80       0.932688    0.932795      0.906089  0.926106
originalPCA95       0.966482    0.899473      0.959697  0.959900

MEJOR RESULTADO:
  Dataset: original
  Modelo: KNN
  F1-Score: 0.9666 ± 0.0298
